# Day 2 – Structured Outputs with Pydantic & LangChain

## Objectives

- Learn Structured Outputs
- Learn Pydantic v2
- Understand Validators
- Learn Field Aliases
- Learn model_config
- Build AI Output Schemas
- Generate Structured Outputs using LangChain
- Parse LLM responses into Python Objects

In [1]:
import sys

print("Python Executable:")
print(sys.executable)

Python Executable:
e:\Udemy 01\Week02\calderr-ai-week2\.venv\Scripts\python.exe


# What are Structured Outputs?

Large Language Models usually generate free-form text.

While this is useful for conversations, production AI systems often require responses in a structured format such as JSON or Python objects.

Structured outputs ensure:

- Consistent responses
- Easy parsing
- Validation
- Reduced hallucinations
- Better integration with software systems

# Why Structured Outputs Matter

Without structured outputs:

User

↓

LLM

↓

Free Text

With structured outputs:

User

↓

LLM

↓

Validated Pydantic Model

↓

Python Object

↓

Application

# Introduction to Pydantic v2

Pydantic is a Python library for data validation and settings management.

It automatically validates data using Python type hints.

Key features:

- Type validation
- Automatic parsing
- JSON serialization
- Nested models
- Validators
- Field aliases
- model_config

In [2]:
from pydantic import BaseModel

# Creating Your First Pydantic Model

A Pydantic model is a Python class that inherits from `BaseModel`. It automatically validates the data based on the type annotations provided for each field.

Advantages:
- Automatic validation
- Automatic type conversion
- Easy serialization
- Clean and structured code

In [3]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    age: int
    cgpa: float

student = Student(
    name="Faizan Ahmad",
    age=20,
    cgpa=3.45
)

print(student)

name='Faizan Ahmad' age=20 cgpa=3.45


# Accessing Model Attributes

Once a Pydantic model is created, its fields can be accessed like normal Python object attributes.

In [4]:
print("Name:", student.name)
print("Age:", student.age)
print("CGPA:", student.cgpa)

Name: Faizan Ahmad
Age: 20
CGPA: 3.45


# Automatic Type Conversion

One of Pydantic's powerful features is automatic type conversion.

If a value can safely be converted to the required type, Pydantic performs the conversion automatically.

In [5]:
student2 = Student(
    name="Ali",
    age="21",      # String
    cgpa="3.82"    # String
)

print(student2)
print(type(student2.age))
print(type(student2.cgpa))

name='Ali' age=21 cgpa=3.82
<class 'int'>
<class 'float'>


# Validation Errors

If the provided data cannot be converted to the required type, Pydantic raises a ValidationError.

This helps catch invalid data before it reaches the application.

In [6]:
from pydantic import ValidationError

try:
    student3 = Student(
        name="Ahmed",
        age="Twenty",
        cgpa=3.4
    )

except ValidationError as e:
    print(e)

1 validation error for Student
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Twenty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


# Default Values

Fields in a Pydantic model can have default values. If a value is not provided during object creation, the default value is used automatically.

In [7]:
class Employee(BaseModel):
    name: str
    department: str = "Artificial Intelligence"
    experience: int = 0

employee = Employee(name="Faizan")

print(employee)

name='Faizan' department='Artificial Intelligence' experience=0


# Exporting a Pydantic Model

Pydantic models can easily be converted into dictionaries or JSON.

This feature is especially useful when sending data through APIs or storing it in databases.

In [8]:
print(employee.model_dump())

print()

print(employee.model_dump_json(indent=4))

{'name': 'Faizan', 'department': 'Artificial Intelligence', 'experience': 0}

{
    "name": "Faizan",
    "department": "Artificial Intelligence",
    "experience": 0
}


# Field Constraints using `Field()`

Pydantic's `Field()` function allows us to define constraints for model fields. These constraints help ensure that the data meets specific requirements before it is accepted.

Common constraints include:

- `gt` : Greater than
- `ge` : Greater than or equal to
- `lt` : Less than
- `le` : Less than or equal to
- `min_length`
- `max_length`

In [9]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(min_length=3, max_length=30)
    price: float = Field(gt=0)
    quantity: int = Field(ge=0)

product = Product(
    name="Laptop",
    price=850.50,
    quantity=5
)

print(product)

name='Laptop' price=850.5 quantity=5


# Custom Validation using `field_validator`

Sometimes built-in constraints are not enough. Pydantic allows us to define custom validation logic using `field_validator`.

In [10]:
from pydantic import field_validator

class User(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def validate_username(cls, value):
        if len(value) < 5:
            raise ValueError("Username must contain at least 5 characters.")
        return value

user = User(username="Faizan")
print(user)

username='Faizan'


# Model Configuration (`model_config`)

Pydantic v2 uses `model_config` to customize model behavior.

Some useful configurations include:

- Automatically stripping whitespace
- Allowing extra fields
- Freezing models
- Populating fields using aliases

In [11]:
from pydantic import ConfigDict

class Employee(BaseModel):

    model_config = ConfigDict(
        str_strip_whitespace=True
    )

    name: str
    department: str

employee = Employee(
    name="   Faizan Ahmad   ",
    department="AI"
)

print(employee)

name='Faizan Ahmad' department='AI'


# Field Aliases

Field aliases allow one field name to accept data using another name.

This is especially useful when APIs use different naming conventions.

In [12]:
class StudentAlias(BaseModel):
    full_name: str = Field(alias="name")
    age: int

student = StudentAlias(
    name="Faizan Ahmad",
    age=20
)

print(student)

full_name='Faizan Ahmad' age=20


# Building Multiple Pydantic Models

Large AI applications usually require multiple schemas for different types of structured data.

Below are five example models.

In [13]:
class Book(BaseModel):
    title: str
    author: str
    pages: int


class Car(BaseModel):
    brand: str
    model: str
    year: int


class JobPosting(BaseModel):
    title: str
    company: str
    location: str
    salary: str
    skills: list[str]


class Patient(BaseModel):
    name: str
    age: int
    disease: str


class Movie(BaseModel):
    title: str
    rating: float
    genre: str

print("Five Pydantic models created successfully!")

Five Pydantic models created successfully!


# Structured Outputs with LangChain

Instead of generating free-form text, LangChain can return structured data that conforms to a predefined Pydantic model.

This makes AI applications more reliable because the output is validated automatically.

In [14]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API Key Loaded:", os.getenv("GROQ_API_KEY") is not None)

API Key Loaded: True


In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM Initialized Successfully!")

LLM Initialized Successfully!


# Preparing for Structured Output Extraction

The `JobPosting` schema created above will be used in Lab 2.1 to extract structured information from unstructured job advertisements.

The extracted information will include:

- Job Title
- Company
- Location
- Salary
- Required Skills

Using a schema ensures that the extracted data is consistent and validated.

In [16]:
print("JobPosting Schema")

job = JobPosting(
    title="AI Engineer",
    company="OpenAI",
    location="Remote",
    salary="$100,000 - $150,000",
    skills=["Python", "LangChain", "Pydantic", "Docker"]
)

print(job.model_dump())

JobPosting Schema
{'title': 'AI Engineer', 'company': 'OpenAI', 'location': 'Remote', 'salary': '$100,000 - $150,000', 'skills': ['Python', 'LangChain', 'Pydantic', 'Docker']}


# Observations

During this exercise, the following observations were made:

- Pydantic simplifies data validation using Python type hints.
- Invalid data is detected automatically.
- Field constraints improve data quality.
- Custom validators allow additional business rules.
- Field aliases provide flexibility when working with external APIs.
- Structured outputs are essential for building reliable AI applications.

# Conclusion

In this notebook, I learned how to use Pydantic v2 to create validated data models for AI applications.

I explored field constraints, custom validators, model configuration, and field aliases. These concepts provide the foundation for generating structured outputs from Large Language Models.

The knowledge gained in this notebook will be applied in Lab 2.1 to build a structured output extractor using LangChain and the Groq API.